# 序列逆置 （加注意力的seq2seq）
使用attentive sequence to sequence 模型将一个字符串序列逆置。例如 `OIMESIQFIQ` 逆置成 `QIFQISEMIO`(下图来自网络，是一个加attentino的sequence to sequence 模型示意图)
![attentive seq2seq](./seq2seq-attn.jpg)

In [2]:
import numpy as np
import tensorflow as tf
import collections
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import layers, optimizers, datasets
import os,sys,tqdm

## 玩具序列数据生成
生成只包含[A-Z]的字符串，并且将encoder输入以及decoder输入以及decoder输出准备好（转成index）

In [3]:
import random
import string

def randomString(stringLength):
    """Generate a random string with the combination of lowercase and uppercase letters """

    letters = string.ascii_uppercase
    return ''.join(random.choice(letters) for i in range(stringLength))

def get_batch(batch_size, length):
    batched_examples = [randomString(length) for i in range(batch_size)]
    enc_x = [[ord(ch)-ord('A')+1 for ch in list(exp)] for exp in batched_examples]
    y = [[o for o in reversed(e_idx)] for e_idx in enc_x]
    dec_x = [[0]+e_idx[:-1] for e_idx in y]
    return (batched_examples, tf.constant(enc_x, dtype=tf.int32), 
            tf.constant(dec_x, dtype=tf.int32), tf.constant(y, dtype=tf.int32))
print(get_batch(2, 10))

(['IRZODHANYD', 'XOVWELBIEA'], <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 9, 18, 26, 15,  4,  8,  1, 14, 25,  4],
       [24, 15, 22, 23,  5, 12,  2,  9,  5,  1]])>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 0,  4, 25, 14,  1,  8,  4, 15, 26, 18],
       [ 0,  1,  5,  9,  2, 12,  5, 23, 22, 15]])>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 4, 25, 14,  1,  8,  4, 15, 26, 18,  9],
       [ 1,  5,  9,  2, 12,  5, 23, 22, 15, 24]])>)


# 建立sequence to sequence 模型

完成两空，模型搭建以及单步解码逻辑

In [4]:
class mySeq2SeqModel(keras.Model):
    def __init__(self):
        super(mySeq2SeqModel, self).__init__()
        self.v_sz=27
        self.hidden = 128
        self.embed_layer = tf.keras.layers.Embedding(self.v_sz, 64, 
                                                    batch_input_shape=[None, None])
        
        self.encoder_cell = tf.keras.layers.SimpleRNNCell(self.hidden)
        self.decoder_cell = tf.keras.layers.SimpleRNNCell(self.hidden)
        
        self.encoder = tf.keras.layers.RNN(self.encoder_cell, 
                                           return_sequences=True, return_state=True)
        self.decoder = tf.keras.layers.RNN(self.decoder_cell, 
                                           return_sequences=True, return_state=True)
        # Projects vectors to hidden size (used both for scoring and fusing context)
        self.dense_attn = tf.keras.layers.Dense(self.hidden)
        # Projection to vocabulary logits
        self.dense = tf.keras.layers.Dense(self.v_sz)
        
        
    @tf.function
    def call(self, enc_ids, dec_ids):
        '''
        带attention的seq2seq：
        1) 编码得到 enc_out (b, T_enc, H) 和 enc_state 作为解码初始状态
        2) 先运行decoder得到 dec_out (b, T_dec, H)
        3) 计算注意力分数: scores = dec_out @ (W*enc_out)^T
        4) 取softmax得到权重并聚合上下文 context (b, T_dec, H)
        5) 融合: h_tilde = dec_out + W*context
        6) 通过输出层得到 logits
        '''
        # Encode
        enc_emb = self.embed_layer(enc_ids)                    # (b, T_enc, E)
        enc_out, enc_state = self.encoder(enc_emb)             # (b, T_enc, H), (b, H)
        
        # Decode (teacher forcing inputs)
        dec_emb = self.embed_layer(dec_ids)                    # (b, T_dec, E)
        dec_out, _ = self.decoder(dec_emb, initial_state=enc_state)  # (b, T_dec, H)
        
        # Bilinear attention: project encoder outputs once
        enc_proj = self.dense_attn(enc_out)                    # (b, T_enc, H)
        # scores: (b, T_dec, T_enc)
        scores = tf.matmul(dec_out, enc_proj, transpose_b=True)
        attn_weights = tf.nn.softmax(scores, axis=-1)          # (b, T_dec, T_enc)
        
        # Context: (b, T_dec, H) using original enc_out as values
        context = tf.matmul(attn_weights, enc_out)             # (b, T_dec, H)
        
        # Fuse context (project context, then add to dec_out)
        fused = dec_out + self.dense_attn(context)             # (b, T_dec, H)
        logits = self.dense(fused)                              # (b, T_dec, V)
        return logits
    
    
    @tf.function
    def encode(self, enc_ids):
        enc_emb = self.embed_layer(enc_ids) # shape(b_sz, len, emb_sz)
        enc_out, enc_state = self.encoder(enc_emb)
        # 初始状态包含一个上下文占位（用最后一个encoder隐状态），以及decoder初始隐状态
        return enc_out, [enc_out[:, -1, :], enc_state]
    
    def get_next_token(self, x, state, enc_out):
        '''
        单步带注意力的解码。
        state = [prev_context, dec_hidden_state]
        '''
        # 取出当前decoder隐状态
        prev_ctx, dec_state = state
        
        # 当前输入token嵌入并一步RNN
        inp_emb = self.embed_layer(x)                          # (b, E)
        h, dec_state = self.decoder_cell.call(inp_emb, dec_state)  # (b, H)
        
        # 计算注意力权重（bilinear）：对encoder输出做一次投影
        enc_proj = self.dense_attn(enc_out)                    # (b, T_enc, H)
        # scores: (b, T_enc) from (b, H) @ (b, T_enc, H)^T
        scores = tf.matmul(tf.expand_dims(h, axis=1), enc_proj, transpose_b=True)  # (b, 1, T_enc)
        scores = tf.squeeze(scores, axis=1)                    # (b, T_enc)
        attn = tf.nn.softmax(scores, axis=-1)                  # (b, T_enc)
        
        # context: (b, H)
        context = tf.matmul(tf.expand_dims(attn, axis=1), enc_out)  # (b, 1, H)
        context = tf.squeeze(context, axis=1)                  # (b, H)
        
        # 融合并投到词表（避免不必要的维度扩展/压缩）
        fused = h + self.dense_attn(context)                   # (b, H)
        logits = self.dense(fused)                             # (b, V)
        out = tf.argmax(logits, axis=-1)
        
        # 新state携带当前context与新的decoder隐状态
        new_state = [context, dec_state]
        return out, new_state

# Loss函数以及训练逻辑

In [5]:
@tf.function
def compute_loss(logits, labels):
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels)
    losses = tf.reduce_mean(losses)
    return losses

@tf.function
def train_one_step(model, optimizer, enc_x, dec_x, y):
    with tf.GradientTape() as tape:
        logits = model(enc_x, dec_x)
        loss = compute_loss(logits, y)

    # compute gradient
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

def train(model, optimizer, seqlen):
    loss = 0.0
    accuracy = 0.0
    for step in range(2000):
        batched_examples, enc_x, dec_x, y = get_batch(32, seqlen)
        loss = train_one_step(model, optimizer, enc_x, dec_x, y)
        if step % 500 == 0:
            print('step', step, ': loss', loss.numpy())
    return loss

# 训练迭代

In [6]:
optimizer = optimizers.Adam(0.0005)
model = mySeq2SeqModel()
train(model, optimizer, seqlen=20)

step 0 : loss 3.3045845
step 500 : loss 0.7438755
step 1000 : loss 0.17522117
step 1500 : loss 0.10773263


<tf.Tensor: shape=(), dtype=float32, numpy=0.065828>

# 测试模型逆置能力
首先要先对输入的一个字符串进行encode，然后在用decoder解码出逆置的字符串

测试阶段跟训练阶段的区别在于，在训练的时候decoder的输入是给定的，而在预测的时候我们需要一步步生成下一步的decoder的输入

In [7]:
def sequence_reversal():
    def decode(init_state, steps, enc_out):
        state0 = tf.nest.flatten(init_state)[0]
        b_sz = tf.shape(state0)[0]
        cur_token = tf.zeros(shape=[b_sz], dtype=tf.int32)
        state = init_state
        collect = []
        for i in range(steps):
            cur_token, state = model.get_next_token(cur_token, state, enc_out)
            collect.append(tf.expand_dims(cur_token, axis=-1))
        out = tf.concat(collect, axis=-1).numpy()
        out = [''.join([chr(idx+ord('A')-1) for idx in exp]) for exp in out]
        return out
    
    batched_examples, enc_x, _, _ = get_batch(32, 20)
    enc_out, state = model.encode(enc_x)
    return decode(state, enc_x.get_shape()[-1], enc_out), batched_examples

def is_reverse(seq, rev_seq):
    rev_seq_rev = ''.join([i for i in reversed(list(rev_seq))])
    if seq == rev_seq_rev:
        return True
    else:
        return False
print([is_reverse(*item) for item in list(zip(*sequence_reversal()))])
print(list(zip(*sequence_reversal())))

[True, False, True, True, False, False, True, True, True, False, False, True, False, True, False, False, True, True, False, False, False, False, False, True, False, False, False, True, True, False, True, True]
[('AWJUEFIQCNJZOVPAIPNI', 'INPIAPVOZJNCQIFEUJWA'), ('OONFRNHQTFOCVSUSKYEQ', 'QEYKSUSVCOFTQHNRFNOM'), ('VJVTDYOZXSENUOVVEBPL', 'LPBEVVOUNESXZOYDTVJV'), ('UTPJDGSTCPGHWQRPAJBD', 'DBJAPRQWHGPCTSGDJPFU'), ('ZLVQNDKSPAXYRETBIKNM', 'MNKIBTERYXAPSKDNQVLZ'), ('ZUAPBCVDVMPAJWHSETEI', 'IETESHWJAPMVDVCBPAUZ'), ('ZYMUNQNPCUPVCYKJOFQZ', 'ZQFOJKYCVPUCPNQNUMYZ'), ('IPEGTXARNLNHMYHZIRPA', 'APRIZHYMHNLNRAXTGEPI'), ('GKMWOEHKSAKVPOLDTFSE', 'ESFTDLOPVKASKHEOWMKG'), ('VSYNXWRDSPBFWNVLWCZL', 'LZCWLVNWFBPSDRWXNYSV'), ('UVUSKLGDUVQRRYPJYNHN', 'NHNYJPYRRQVUDGLKSUVA'), ('PAPTJDTDQVBUUPABMIHS', 'SHIMBAPUUBVQDTDJTPAP'), ('IVTEFQWLLUMBFLNUTYLH', 'HLYTUNLFBMULLWQFETVI'), ('JKNIIANQLGXYLMCEIEHJ', 'JHEIECMLYXGLQNAIINKJ'), ('AWHMAWYMBRTJAHFTXIQD', 'DQIXTFHAJTRBMYWAMHZA'), ('UPRXLDSWNULCFDIUTSRK', 'KRSTUIDFCLUNW